In [1]:
%%capture
%cd ../..

In [2]:
import psycopg

In [3]:
conn = psycopg.connect('postgresql://USER@localhost:5432/mlflow_db')

In [4]:
def decode_if_bytes(x):
    if isinstance(x, bytes):
        return x.decode("utf-8")
    return x

In [5]:
def execute_query(query, conn, limit=None):
    if limit:
        query += f" LIMIT {limit};"
    with conn.cursor() as cursor:
        cursor.execute(query)
        return cursor.fetchall()

In [25]:
experiment_id, experiment_name = "47", "transformers_improved_2"

In [26]:
integral_query = f"""
SELECT
    r.run_uuid,
    p.value AS expression,
    SUM(m.value) AS sum_val_loss
FROM runs r
JOIN params p
    ON r.run_uuid = p.run_uuid
   AND p.key = 'expression'
JOIN metrics m
    ON r.run_uuid = m.run_uuid
   AND m.key = 'val_loss_step'
WHERE r.experiment_id = {experiment_id}
GROUP BY r.run_uuid, p.value
ORDER BY p.value
"""

In [27]:
integral_results = execute_query(integral_query, conn)

In [28]:
import pandas as pd
df = pd.DataFrame(integral_results, columns=["run_uuid", "expression", "val_loss_step_AOC"])
df["expression"] = df["expression"].apply(decode_if_bytes)
df["run_uuid"] = df["run_uuid"].apply(decode_if_bytes)

In [29]:
df.to_csv(f"../analysis/{experiment_name}_AOC.csv", index=False)

In [30]:
experiment_name

'transformers_improved_2'